# Train Duck’s crop-sequence model

Run this in the **NVIDIA A30** Jupyter kernel with `train_rotation.py` in the same folder. Duck’s chat router and voice model are separate. This run benchmarks the next USDA mapped crop class in Minnesota. The underlying USDA polygons were delineated from the full release window, so this is a retrospective benchmark. Operational forecast validation needs boundaries defined before the test season.

[USDA CSB releases](https://www.nass.usda.gov/research-and-science/crop-sequence-boundaries)

In [ ]:
from pathlib import Path
import torch
from train_rotation import OUTPUT, run

assert torch.cuda.is_available(), "Select the NVIDIA A30 kernel."
device = torch.device("cuda")
print(torch.cuda.get_device_name(device))

## Validate, train, and compare

The script checks the year fields and crop labels, filters missing histories, and checks the sizes and separation of county groups before training. It holds out the newest crop year and entire counties. Review both accuracy and macro recall, which gives each observed crop class equal weight.

In [ ]:
report = run(device)
print(f"{report['fields']:,} usable fields, {report['skipped_rows']:,} excluded rows")
print(f"{report['train_counties']} training counties, {report['test_counties']} held-out counties")
print(f"Test year {report['test_year']} across {report['test_examples']:,} held-out fields")
for method, scores in report["metrics"].items():
    print(f"{method:24} accuracy {scores['accuracy']:.2%}  macro recall {scores['macro_recall']:.2%}")
print("Candidate beats both baselines on both metrics:", report["candidate_beats_baselines"])
print("Checkpoint and report:", OUTPUT / report["version"][:16])

## Refreshing data

The production dashboard reads publisher data on request. For the training feed, run `python train_rotation.py --watch` in the CUDA environment to check daily while the process is running. Unchanged sampled data reuse the previous report. Changed data train and evaluate a versioned candidate. This script writes model weights and evaluation summaries, and keeps CSB input rows in memory. Candidate checkpoints require a deployment integration and a review against the current model.

The existing feed covers Minnesota, 2017–2024. The watcher checks this specific service. An annual USDA release at a new URL needs a connector update. Global rotations require local crop records and on-farm outcomes. Hyperspectral interpretation needs tested science-file extraction, and a recommendation model needs yield, water, costs and pest outcomes tied to verified management records.